# 10年定着予測 - 最終ブレンドの再現（66_）

## このノートブックの役割

**提出ファイル `20260816_pool_tabpfn_blend_w70.csv`（Public 0.508793、現最良）を再現する。**

この提出は単一のノートブックの出力ではなく、**先行する6本のノートブックが保存した
Test予測を組み合わせたもの**である。提出プログラムと出力結果を対応させるため、
その合成手順を本ノートブックに明文化する。

## レシピ

```
最終予測 = 0.70 × AutoGluonプールC + 0.30 × TabPFN(top150)
```

### 材料1: AutoGluonプールC（保存済み8実行の単純平均、追加学習なし）

| 由来ノートブック | 出力ファイル | 備考 |
|---|---|---|
| `50_autogluon_memofix` | `AG50_full441_weighted.csv` | 単発Public 0.513108（旧最良）|
| `51_autogluon_catboost_bias` | `AG51_full441_weighted.csv` | 単発Public 0.514263 |
| `53_autogluon_dystack` | `AG53_full441_weighted.csv` | DyStack有効。未提出 |
| `61_autogluon_extended_time` | `AG61_full441_weighted.csv` | time_limit 6時間。未提出 |
| `62_autogluon_seed_averaging` | `AG62_bag16_weighted.csv` | num_bag_folds=16 |
| `62_autogluon_seed_averaging` | `full441_seed42_weighted_testpreds.npy` | シード42 |
| `62_autogluon_seed_averaging` | `full441_seed2024_weighted_testpreds.npy` | シード2024 |
| `62_autogluon_seed_averaging` | `full441_seed7_weighted_testpreds.npy` | シード7 |

すべて **441列・WeightedEnsemble・`49_`のメモパーサー修正後**の実行。
`56_autogluon_lm_block` は特徴量が444列（LMブロック込み）で異質なため**除外**している。

### 材料2: TabPFN(top150)

| 由来ノートブック | 出力ファイル | 備考 |
|---|---|---|
| `63_tabpfn_ensemble` | `tabpfn_top150_testpreds.npy` | TabPFN v2、CatBoost重要度上位150列、3シード平均 |

### 重み w_AG = 0.70 の根拠

`63_`のブレンド重み走査で、単層CatBoost相手には w=0.5 が最適だった。
ただし曲線は極めて平坦（w=0.5で0.502252、w=0.7で0.502838、差+0.0006のみ）。
ブレンド相手をAutoGluonプール（単層CatBoostより強い）に替えるため重みを0.70に上げた。
**Publicで走査して決めた値ではない**（Private評価に対する過学習を避けるため）。

## 実行に必要な前提

先行ノートブック `50_` / `51_` / `53_` / `61_` / `62_` / `63_` が実行済みで、
その出力が `data/output/` 配下に保存されていること。
本ノートブックは**モデルの再学習を一切行わない**（保存済み予測の合成のみ、数秒で完了）。
---

## ライセンス表記（SIGNATE参加規約 第2条8項 / Prior Labs License 第10条）

本提出物は **TabPFN v2**（`tabpfn==2.2.1`）を構成要素として含む。

> **Built with PriorLabs-TabPFN**

`tabpfn` v2 のライセンスは **Prior Labs License v1.1**（Apache 2.0 の派生。第10条で帰属表示を追加）で、
**商用利用が許可されている**ため参加規約 第2条8項（商業利用が禁止されているOSSの利用禁止）に適合する。

⚠️ `tabpfn` の **2.5 / 2.6 / 3系（PyPI の 6.x〜8.x）は非商用ライセンス**であり、
第2条8項に抵触するため使用してはならない。`63_` のインストールセルで `tabpfn==2.2.1` に固定し、
実行時にもバージョンを検証している。

その他の依存（autogluon.tabular / catboost / scikit-learn / pandas / numpy）はいずれも
Apache-2.0 または BSD-3-Clause で商用利用可。詳細は `SUBMISSION_README.md` 第6節を参照。


In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


In [ ]:
import datetime
import numpy as np
import pandas as pd

from common.utils.logger import get_logger

SCRIPT_NAME = "66_final_blend"
TODAY = datetime.datetime.now().strftime("%Y%m%d")
logger = get_logger(SCRIPT_NAME, log_dir=str(PROJECT_ROOT / "logs"))

OUT_ROOT = PROJECT_ROOT / "data" / "output"
OUTPUT_DIR = OUT_ROOT / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ID_COL = "社員ID"
TARGET_COL = "10年定着ラベル"
W_AG = 0.70          # AutoGluonプール側の重み（63_の走査に基づく。Publicでは走査していない）

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"出力先        = {OUTPUT_DIR}")


## 1. 材料の読み込み（由来を明示する）

In [ ]:
# 提出済みCSV(Public 0.508793)を生成したときの入力ファイルを**ファイル名で固定**する。
# glob で「最新」を拾う実装にすると、同じノートブックを再実行して新しい日付の出力が
# 増えたときに材料が入れ替わり、提出物を再現できなくなる（例: 61_ の再実行）。
PINNED = {
    "50_autogluon_memofix":        "20260813/20260813_50_autogluon_memofix_AG50_full441_weighted.csv",
    "51_autogluon_catboost_bias":  "20260814/20260814_51_autogluon_catboost_bias_AG51_full441_weighted.csv",
    "53_autogluon_dystack":        "20260815/20260815_53_autogluon_dystack_AG53_full441_weighted.csv",
    "61_autogluon_extended_time":  "20260815/20260815_61_autogluon_extended_time_AG61_full441_weighted.csv",
    "62_bag16":                    "20260815/20260815_62_autogluon_seed_averaging_AG62_bag16_weighted.csv",
    "62_seed42":                   "20260815/20260815_62_autogluon_seed_averaging_full441_seed42_weighted_testpreds.npy",
    "62_seed2024":                 "20260815/20260815_62_autogluon_seed_averaging_full441_seed2024_weighted_testpreds.npy",
    "62_seed7":                    "20260815/20260815_62_autogluon_seed_averaging_full441_seed7_weighted_testpreds.npy",
    "63_tabpfn_top150":            "20260816/20260816_63_tabpfn_ensemble_tabpfn_top150_testpreds.npy",
}


def _resolve(key, pattern):
    """固定ファイル名を優先。無ければglobで探すが、その場合は再現性が保証されない旨を警告する。"""
    pin = OUT_ROOT / PINNED[key]
    if pin.exists():
        return pin, True
    fs = sorted(OUT_ROOT.glob(pattern))
    if not fs:
        return None, False
    print(f"  ⚠️ {key}: 固定ファイル {PINNED[key]} が無いため {fs[-1].name} を使用。"
          f"提出済みCSVとは一致しない可能性がある")
    return fs[-1], False


def load_csv(key, pattern):
    f, _ = _resolve(key, pattern)
    if f is None:
        return None, None
    return pd.read_csv(f, header=None, names=[ID_COL, "p"]).set_index(ID_COL)["p"], f.name


def load_npy(key, pattern):
    f, _ = _resolve(key, pattern)
    return (np.load(f), f.name) if f is not None else (None, None)


# --- 材料1: AutoGluonプールC の構成要素（由来ノートブックを明記）---
AG_SOURCES = [
    ("50_autogluon_memofix",        "50_autogluon_memofix",       "csv", "*/*_50_autogluon_memofix_AG50_full441_weighted.csv"),
    ("51_autogluon_catboost_bias",  "51_autogluon_catboost_bias", "csv", "*/*_51_autogluon_catboost_bias_AG51_full441_weighted.csv"),
    ("53_autogluon_dystack",        "53_autogluon_dystack",       "csv", "*/*_53_autogluon_dystack_AG53_full441_weighted.csv"),
    ("61_autogluon_extended_time",  "61_autogluon_extended_time", "csv", "*/*_61_autogluon_extended_time_AG61_full441_weighted.csv"),
    ("62_autogluon_seed_averaging", "62_bag16",                   "csv", "*/*_62_autogluon_seed_averaging_AG62_bag16_weighted.csv"),
    ("62_autogluon_seed_averaging", "62_seed42",                  "npy", "*/*_62_autogluon_seed_averaging_full441_seed42_weighted_testpreds.npy"),
    ("62_autogluon_seed_averaging", "62_seed2024",                "npy", "*/*_62_autogluon_seed_averaging_full441_seed2024_weighted_testpreds.npy"),
    ("62_autogluon_seed_averaging", "62_seed7",                   "npy", "*/*_62_autogluon_seed_averaging_full441_seed7_weighted_testpreds.npy"),
]

# 社員IDの並びは、CSVを持つ最初の材料から取る（以降すべてこの並びに揃える）
IDX, _ = load_csv(AG_SOURCES[0][1], AG_SOURCES[0][3])
assert IDX is not None, "50_ の提出CSVが見つからない。先に 50_ を実行すること"
IDX = IDX.index

rows, ag_preds = [], []
for nb_name, key, kind, pat in AG_SOURCES:
    if kind == "csv":
        ser, fname = load_csv(key, pat)
        v = None if ser is None else ser.reindex(IDX).values
    else:
        v, fname = load_npy(key, pat)
    assert v is not None, f"{nb_name} の出力が見つからない: {pat}"
    assert len(v) == len(IDX), f"{fname}: 行数が {len(v)} で IDX({len(IDX)}) と違う"
    assert not np.isnan(v).any(), f"{fname}: 欠損がある"
    ag_preds.append(v)
    rows.append({"材料": "AutoGluonプールC", "由来ノートブック": nb_name,
                 "ファイル": fname, "予測平均": round(float(v.mean()), 4)})

POOL = np.mean(ag_preds, axis=0)

# --- 材料2: TabPFN(top150) ---
TP, tp_name = load_npy("63_tabpfn_top150", "*/*_63_tabpfn_ensemble_tabpfn_top150_testpreds.npy")
assert TP is not None, "63_ の TabPFN top150 予測が見つからない。先に 63_ を実行すること"
assert len(TP) == len(IDX) and not np.isnan(TP).any()
rows.append({"材料": "TabPFN(top150)", "由来ノートブック": "63_tabpfn_ensemble",
             "ファイル": tp_name, "予測平均": round(float(TP.mean()), 4)})

prov = pd.DataFrame(rows)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 78)
print(prov.to_string(index=False))
print()
print(f"AutoGluonプールC = 上記{len(ag_preds)}本の単純平均  予測平均={POOL.mean():.4f}")
print(f"TabPFN(top150)                                   予測平均={TP.mean():.4f}")
print(f"相関(TabPFN, プールC) = {np.corrcoef(TP, POOL)[0,1]:.4f}  MAD = {np.abs(TP-POOL).mean():.5f}")


## 2. 合成と保存

In [ ]:
blend = W_AG * POOL + (1.0 - W_AG) * TP

assert blend.shape == (len(IDX),)
assert np.isfinite(blend).all()
assert (blend > 0).all() and (blend < 1).all(), "確率が[0,1]の外に出ている"

out_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_pool8_tabpfn_w{int(W_AG*100)}.csv"
pd.DataFrame({ID_COL: IDX, TARGET_COL: blend}).to_csv(out_path, index=False, header=False)
print(f"保存: {out_path}")
print(f"  行数={len(blend)}  予測平均={blend.mean():.4f}  [min {blend.min():.4f}, max {blend.max():.4f}]")

# 材料の由来表も一緒に残す（提出プログラムと出力の対応を追えるようにする）
prov.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_provenance.csv", index=False)
logger.info(f"最終ブレンドを保存: {out_path.name}")


## 3. 既提出ファイルとの一致確認

`20260816_pool_tabpfn_blend_w70.csv`（Public 0.508793）が手元にある場合、
本ノートブックの出力がそれと一致することを確認する。
**一致すれば「提出したCSV = このプログラムの出力」が証明される。**


In [ ]:
_refs = sorted(OUT_ROOT.glob("*/20260816_pool_tabpfn_blend_w70.csv"))
ref = (pd.read_csv(_refs[-1], header=None, names=[ID_COL, "p"]).set_index(ID_COL)["p"]
       if _refs else None)
ref_name = _refs[-1].name if _refs else None
if ref is None:
    print("参照ファイルが見つからないので照合をスキップ（新規に作った場合はこれで正常）")
else:
    r = ref.reindex(IDX).values
    d = float(np.abs(blend - r).max())
    print(f"参照: {ref_name}（Public 0.508793）")
    print(f"最大絶対差 = {d:.3e}")
    assert d < 1e-9, f"提出済みファイルと一致しない（最大差 {d:.3e}）。材料かレシピが違う"
    print("✅ 完全一致。提出したCSVはこのプログラムで再現できる")


## 5. hire_fixed基盤モデルとの追加プール（`70_`由来）

`70_hire_fixed_fm_submission`が生成したTabPFN・TabICL（`hire_fixed`=78列、TabDPTは除く。
第104節でTabDPTを含めるとPublicが悪化すると確認済み）を使い、
既存の`blend`（TabPFN top150 版、Public 0.508793）ともう一つの独立したブレンドを
50:50で平均する。両者は同じAutoGluonプールC(70%)を共有しているので、
実質的には**プールC 70% : TabPFN(top150) 15% : [TabPFN+TabICL(hire_fixed)平均] 15%**
という3方向ブレンドになる。

Jensenの不等式（loglossは確率について凸）より、期待loglossは`blend`と`blend_fm2`の
単純平均以下になることが理論的に保証される（[[private-lb-variance-strategy]]）。


In [ ]:
# --- 材料3: TabPFN(hire_fixed) / TabICL(hire_fixed)（70_の生予測、TabDPTは除く） ---
PINNED_HF = {
    "70_tabpfn_hire_fixed": "20260816/20260816_70_hire_fixed_fm_submission_tabpfn_hire_fixed_testpreds.npy",
    "70_tabicl_hire_fixed": "20260816/20260816_70_hire_fixed_fm_submission_tabicl_hire_fixed_testpreds.npy",
}


def _load_pinned_npy(key, pinned_map):
    path = OUT_ROOT / pinned_map[key]
    assert path.exists(), f"{key}: 固定ファイルが見つからない: {path}（先に70_を実行すること）"
    v = np.load(path)
    assert len(v) == len(IDX), f"{key}: 行数が {len(v)} で IDX({len(IDX)}) と違う"
    assert not np.isnan(v).any(), f"{key}: 欠損がある"
    return v


tabpfn_hf = _load_pinned_npy("70_tabpfn_hire_fixed", PINNED_HF)
tabicl_hf = _load_pinned_npy("70_tabicl_hire_fixed", PINNED_HF)
FM2_hire_fixed = np.mean([tabpfn_hf, tabicl_hf], axis=0)

print(f"TabPFN(hire_fixed)  予測平均={tabpfn_hf.mean():.4f}")
print(f"TabICL(hire_fixed)  予測平均={tabicl_hf.mean():.4f}")
print(f"FM2(hire_fixed)平均 予測平均={FM2_hire_fixed.mean():.4f}")

blend_fm2 = W_AG * POOL + (1.0 - W_AG) * FM2_hire_fixed
d_ref_fm2 = np.abs(blend_fm2 -
    pd.read_csv(sorted(OUT_ROOT.glob("*/20260816_70_hire_fixed_fm_submission_pool_fm2_hire_fixed_w70.csv"))[-1],
                header=None, names=[ID_COL, "p"]).set_index(ID_COL)["p"].reindex(IDX).values).max()
print(f"\nblend_fm2 と 70_の提出済みfm2ファイルとの最大絶対差 = {d_ref_fm2:.3e}")
assert d_ref_fm2 < 1e-9, "70_の提出物と一致しない"
print("✅ blend_fm2 は 70_ の pool_fm2_hire_fixed_w70.csv（Public 0.509339）と完全一致")

# --- pooled: top150ブレンドとhire_fixedブレンドの単純平均 ---
pooled = 0.5 * blend + 0.5 * blend_fm2
assert np.isfinite(pooled).all() and (pooled > 0).all() and (pooled < 1).all()

pooled_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_pool_top150_hire_fixed_avg.csv"
pd.DataFrame({ID_COL: IDX, TARGET_COL: pooled}).to_csv(pooled_path, index=False, header=False)
print(f"\n保存: {pooled_path}")
print(f"  予測平均={pooled.mean():.4f}  相関(blend, blend_fm2)={np.corrcoef(blend, blend_fm2)[0,1]:.4f}"
      f"  MAD={np.abs(blend-blend_fm2).mean():.5f}")

# 手作業で作った既存ファイルとの一致確認（あれば）
_manual = sorted(OUT_ROOT.glob("*/20260816_pool_top150_hire_fixed_avg.csv"))
if _manual:
    d = np.abs(pooled - pd.read_csv(_manual[-1], header=None, names=[ID_COL, "p"])
               .set_index(ID_COL)["p"].reindex(IDX).values).max()
    print(f"\n手作業版({_manual[-1].name})との最大絶対差 = {d:.3e}")
    assert d < 1e-9, "手作業版と一致しない"
    print("✅ 完全一致。手作業で作ったファイルはこのプログラムで再現できる")

logger.info(f"hire_fixedプールを保存: {pooled_path.name}")


## 4. 提出プログラムとしての位置づけ

### 出力1: `pool8_tabpfn_w70`（Public 0.508793、現最良）を再現するのに必要なノートブック

| 順序 | ノートブック | 役割 |
|---|---|---|
| 1 | `50_autogluon_memofix` | AutoGluon 441列（`49_`のパーサー修正込み）|
| 2 | `51_autogluon_catboost_bias` | 同上 + NN_TORCH/KNN除外 |
| 3 | `53_autogluon_dystack` | 同上 + DyStack有効 |
| 4 | `61_autogluon_extended_time` | 同上 + time_limit 6時間 |
| 5 | `62_autogluon_seed_averaging` | 同上をシード3本 + bag16 |
| 6 | `63_tabpfn_ensemble` | TabPFN v2（重要度上位150列、3シード）|
| 7 | **`66_final_blend`（本ノートブック、第2節）** | **上記の予測を 0.70:0.30 で合成** |

### 出力2: `pool_top150_hire_fixed_avg`（Public未確認、`fm2`単体0.509339）を再現するのに必要な追加ノートブック

上記1〜6に加えて:

| 順序 | ノートブック | 役割 |
|---|---|---|
| 8 | `68_foundation_models_3way` | TabPFN/TabICL/TabDPT×5特徴量セットの検証（`hire_fixed`を特定）|
| 9 | `70_hire_fixed_fm_submission` | TabPFN・TabICL・TabDPTを`hire_fixed`(78列)でTest予測 |
| 7' | **`66_final_blend`（本ノートブック、第5節）** | **出力1のblendと、TabPFN+TabICL(hire_fixed)の
    AutoGluonブレンドを50:50で平均** |

1〜6・8〜9 はいずれも同じ特徴量パイプライン（441列、ブロックL2、メモパーサー修正済み。
`68_`/`70_`はLM無し版）を内部に持つ。各ノートブックは独立に実行でき、出力は
`data/output/<日付>/` に保存される。

### 補足: 特徴量パイプラインの実体

1〜6 のノートブックはすべて、冒頭で同一の特徴量生成コードを持つ（各ノートブックが自己完結する設計）。
その系譜は `12_` 〜 `18_`（基本集約・TF-IDF・四半期特徴量）→ `28_`（ブロックL2）→
`49_`（メモパーサーのカバレッジ修正）であり、詳細は `data/output/submit_result_report.md` を参照。
